### 1. 非对话模型
不提了
### 2. 对话模型
未过时，仍是官方主推的方式
#### 2.1 init_chat_model
根据模型ID自动推断，环境变量需要有相应的平替API_KEY，找不到则失败
Deepseek模型会被推断为深度求索的API，需要单独的 langchain-deepseek 包
#### 2.2 ChatOpenAI
ChatOpenAI可以接收所有OpenAICompatible的API调用，如果非官方API，则需要在环境变量中提供OEPNAI_BASE_URL



In [6]:
import dotenv

from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage

import os

dotenv.load_dotenv()

########核心代码############
chat_model = init_chat_model(
    api_key=os.getenv("OPENAI_API_KEY"),  # 安全读取
    # base_url=os.getenv("OPENAI_BASE_URL"),
    model="deepseek-chat"
)
print("start")
messages = [
    SystemMessage(content="我是人工智能助手，我叫小智"),
    HumanMessage(content="你好，小智，我是小明，很高兴认识你，一句话回答我")
]

response = chat_model.invoke(messages) # 输入消息列表

print(type(response))  # <class 'langchain_core.messages.ai.AIMessage'>
print(response.content)

start
<class 'langchain_core.messages.ai.AIMessage'>
你好小明，我也很高兴认识你！


In [7]:
#导入 dotenv 库的 load_dotenv 函数，用于加载环境变量文件（.env）中的配置
import dotenv
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()  #加载当前目录下的 .env 文件

# 创建大模型实例
llm = ChatOpenAI(model="deepseek-chat")  # 默认使用 gpt-3.5-turbo

# 直接提供问题，并调用llm
response = llm.invoke("什么是大模型？一句话")
print(response)

content='大模型是指拥有海量参数（通常达数十亿甚至万亿级别）的深度学习模型，能够通过大规模数据训练处理复杂任务（如自然语言理解、生成、多模态交互等）。' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 9, 'total_tokens': 50, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 9}, 'model_provider': 'openai', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache', 'id': '1a4ddf72-ed60-4433-9561-7f17ad753e08', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019cdc35-88b2-7051-bd4a-c707d3f5f6ed-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 9, 'output_tokens': 41, 'total_tokens': 50, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}


### 2. 提示词模板


In [8]:
from langchain_core.prompts import ChatPromptTemplate

# 需要注意的一点是，这里需要指明具体的role，在这里是system和用户
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是世界级的技术文档编写者"),
    ("user", "{input}")  # {input}为变量
])

# 我们可以把prompt和具体llm的调用和在一起。
chain = prompt | llm
message = chain.invoke({"input": "大模型中的LangChain是什么?"})
print(message)

# print(type(message))

content='LangChain 是一个用于开发由大型语言模型（LLMs）驱动的应用程序的**开源框架**。它的核心思想是**“链接”** 各种组件，将LLM的强大能力与外部数据源、计算工具和逻辑流程结合起来，从而构建出功能强大、实用的应用。\n\n简单来说，LangChain 让大模型不再是一个只会聊天的“孤岛”，而是变成了一个可以**行动、思考和利用外部知识**的智能体。\n\n### LangChain 的核心价值与解决的问题：\n\n1.  **连接能力**：LLM本身的知识是静态的（基于训练数据截止日期），且无法直接访问外部系统。LangChain 提供了标准化的接口和工具，让LLM可以轻松地：\n    *   **读取外部文档和数据**（如PDF、数据库、API）。\n    *   **使用工具**（如计算器、搜索引擎、代码执行环境）。\n    *   **与环境互动**（如通过API操作软件）。\n\n2.  **上下文管理**：LLM有输入长度限制。LangChain 提供了智能的上下文窗口管理，例如通过“检索增强生成”技术，只从海量资料中检索出最相关的片段提供给LLM，从而突破上下文限制。\n\n3.  **链式编排**：将复杂的任务分解为多个步骤，并按顺序或条件执行。例如：“获取用户问题 -> 搜索网络 -> 总结搜索结果 -> 生成最终答案”。这个多步骤的工作流就是一条“链”。\n\n### LangChain 的主要组件：\n\n可以把它想象成一个构建智能应用的“乐高工具箱”：\n\n*   **模型 I/O**：与各种LLM（如OpenAI GPT、Anthropic Claude、开源模型）交互的核心接口，统一了调用方式。\n*   **检索**：从外部数据源（向量数据库、文档存储）中查找和获取与问题相关的信息。这是实现“基于自有知识库问答”的关键。\n*   **记忆**：让应用在多次交互中记住之前的对话历史或上下文，实现连贯的对话。\n*   **代理**：这是LangChain最强大的概念之一。一个“代理”就像一个拥有LLM“大脑”的机器人，它可以**自主决定**使用哪些工具（如搜索、计算、查数据库）来完成任务。你告诉它目标，它自己规划步骤。\n*   **链**：将上述组件按预定顺序组合起来，完成特定任务。链可以是简单的

In [ ]:
from langchain_classic.memory import ChatMessageHistory

history = ChatMessageHistory()

history.add_ai_message("我是一个无所不能的小智")
history.add_user_message("你好，我叫小明，请介绍一下你自己")
history.add_user_message("我是谁呢？")

print(history.messages)  #返回List[BaseMessage]类型

In [ ]:
from langchain_classic.prompts.chat import ChatPromptTemplate
from langchain_classic.prompts.prompt import PromptTemplate
from langchain_community.chat_models import ChatOllama

# 构建模版
template = "你是一个有用的助手，可以将{input_language}翻译成{output_language}。"
human_template = "{text}"

# 生成对话形式的聊天信息格式
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", template),
    ("human", human_template),
])

# 格式化变量输入
messages = chat_prompt.format_messages(input_language="中文", output_language="英语", text="我爱编程")

# 实例化Ollama启动的模型
ollama_llm = ChatOllama(model="deepseek-r1:7b")

# 执行推理
result = ollama_llm.invoke(messages)

print(result.content)